In [1]:
from top import Top

t = Top()
t.start()


In [ ]:
import tensorflow as tf


In [ ]:
data = tf.keras.datasets.fashion_mnist
(training_images, training_labels), (test_images, test_labels) = data.load_data()


In [ ]:
training_images.shape, test_images.shape


In [ ]:
labels = ["t-shirt/haut", "pantalon", "pull", "robe", "manteau", "sandales", "chemise",
"baskets", "sac", "bottines"]


In [ ]:
labels[8]


In [ ]:
import matplotlib.pyplot as plt

# Choisir training ou test images en activant LA ligne ad'hoc
img_ids, img_labels = training_images, training_labels
# img_ids, img_labels = test_images, test_labels

# Calculate the number of rows and columns for the subplot grid
n_images = 20
n_cols = 10  # You can adjust the number of columns
n_rows = (n_images + n_cols - 1) // n_cols

plt.figure(figsize=(n_cols * 1.5, n_rows * 1.5)) # Adjust figure size as needed to accommodate xlabel

print("Images " +  ("d'entrainement" if img_labels.size== 60000 else 'de test'))

for i in range(n_images):
    plt.subplot(n_rows, n_cols, i + 1) # Create a subplot for each image
    plt.imshow(img_ids[i], cmap='gray')
    plt.xlabel(str(i) + ' - ' + labels[img_labels[i]].capitalize()) # Use xlabel for title below the image
    plt.xticks([]) # Remove x ticks
    plt.yticks([]) # Remove y ticks
    plt.axis('on') # Turn axis on to show xlabel

plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show()


In [ ]:
# ==========================================================
# 1️⃣  Fixer la graine pour rendre les résultats reproductibles
# ==========================================================
import tensorflow as tf
import numpy as np
import random
import os

# Fixer la graine aléatoire (Pour avoir des résultats similaire avec un même param patience, appel& plusieurs fois)
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

# ==========================================================
# 2️⃣  Imports et préparation des données
# ==========================================================
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping


In [ ]:
ini_shape = training_images.shape
f_training_images = training_images.reshape((60000, 28*28)) # f_... comme flat_
f_test_images = test_images.reshape(-1, 28*28) # -1 = toutes les images du dataset

print('Applatissement des imges (Flatten): 28 x 28 = 784')
print('shape: ', ini_shape, ' → ', f_training_images.shape)

# Normalisation des données (moved from cell uf0-GJSPFcdl)
f_training_images = f_training_images / 255.0
f_test_images = f_test_images / 255.0


In [ ]:
ini_shape = training_labels.shape
ini_label_img45 = training_labels[45]

training_labels = tf.keras.utils.to_categorical(training_labels)
test_labels = tf.keras.utils.to_categorical(test_labels)

print('Classification (One Hot Encoding)')
print('shape: ', ini_shape, ' → ', training_labels.shape)
print('Classe img #45:', ini_label_img45, ' → ', training_labels[45])


In [ ]:
import matplotlib.pyplot as plt

def graph4val(history, name='Early Stopping'):

    # Récupération des métriques
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    # Création de la figure
    plt.figure(figsize=(6, 4))

    # Accuracy
    plt.plot(epochs, acc, 'r-', label='Accuracy entraînement')
    plt.plot(epochs, val_acc, 'r--', label='Accuracy validation')

    # Loss
    plt.plot(epochs, loss, 'b-', label='Loss entraînement')
    plt.plot(epochs, val_loss, 'b--', label='Loss validation')

    # Mise en forme
    name = name.replace(" ", r"\ ")
    plt.title(rf"Evolution (accuracy & loss) $\mathbf{{{name}}}$")
    plt.xlabel("Epochs")
    plt.ylabel("Values")
    plt.legend(loc="center right")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Test avec différentes valeursde la patience du Earliy Stopping

* patience in [3, 5, 7, 10, 12, 15, 21, 25]

In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import pandas as pd # Ajout de l'import pour pandas

# Définir la liste des valeurs de patience
patience_values = [3, 5, 7, 10, 12, 15, 21, 25]
# patience_values = [1, 2, 3]
# Obtenir la valeur de patience maximale pour l'exécution d'entraînement unique
max_patience = max(patience_values)

# Normalisation des données (déplacé hors de la boucle)
# Vérifier si la normalisation a déjà été effectuée pour éviter de la répéter.
if np.max(f_training_images) > 1.0 or np.max(f_test_images) > 1.0:
    f_training_images = f_training_images / 255.0
    f_test_images = f_test_images / 255.0
    print("Données normalisées.")
else:
    print("Données déjà normalisées.")

# Dictionnaire pour stocker les résultats (sera rempli après l'entraînement)
results_summary = {}

print(f"\n--- Entraînement avec patience maximale = {max_patience} ---")
# Utiliser la patience maximale pour le nom de fichier de l'exécution d'entraînement unique
best_model_file = f"best_model_{max_patience}.h5"
history_file = f"training_history_{max_patience}.pkl"


# Construire et compiler le modèle (toujours repartir de zéro pour cette approche)
model = Sequential([
    Dense(units=784, activation="relu", input_shape=(784,)),
    tf.keras.layers.Dropout(0.5),
    Dense(units=128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    Dense(units=10, activation="softmax"),
])
model.compile(
    loss="categorical_crossentropy",
    optimizer=SGD(learning_rate=0.01),
    metrics=["accuracy"]
)

# Définir les callbacks en utilisant la patience maximale
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=max_patience, restore_best_weights=True)
model_ckp = ModelCheckpoint(
    filepath=best_model_file,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True
)

# Entraîner le modèle une seule fois avec la patience maximale
h = model.fit(
    f_training_images, training_labels,
    epochs=1000, # Nombre élevé d'époques, EarlyStopping l'arrêtera
    validation_data=(f_test_images, test_labels),
    callbacks=[model_ckp, stop]
)

# Sauvegarder l'historique de l'entraînement
with open(history_file, 'wb') as f:
    pickle.dump(h, f)

print(f"\nEntraînement avec patience maximale ({max_patience}) terminé.")
print(f"Historique sauvegardé dans {history_file}")
print(f"Meilleur modèle sauvegardé dans {best_model_file}")

# Maintenant, traiter l'historique pour trouver les résultats pour toutes les valeurs de patience
print("\n--- Analyse de l'historique pour différentes valeurs de patience ---")

# Charger l'historique sauvegardé
with open(history_file, 'rb') as f:
    history = pickle.load(f)

# Extraire l'accuracy et la loss de validation
val_accuracy = history.history['val_accuracy']
val_loss = history.history['val_loss']
epochs = range(1, len(val_accuracy) + 1)

# Trouver les meilleurs résultats pour chaque valeur de patience
for patience in patience_values:
    print(f"\nAnalyse pour patience = {patience}")
    best_val_acc_for_p = -1
    best_epoch_for_p = -1
    best_val_loss_for_p = -1
    last_best_epoch = 0 # Garder une trace de l'époque avec la meilleure val_accuracy rencontrée jusqu'à présent

    for i in range(len(val_accuracy)):
        current_val_acc = val_accuracy[i]

        # Mettre à jour la meilleure accuracy rencontrée jusqu'à présent et son époque
        if current_val_acc > val_accuracy[last_best_epoch]:
            last_best_epoch = i

        # Vérifier si l'arrêt anticipé se déclencherait à la fin de la fenêtre de patience
        # L'arrêt anticipé se déclenche si la val_accuracy à l'époque 'i' n'est pas meilleure que
        # le maximum de val_accuracy dans la fenêtre [i-patience, i-1].
        # Si l'époque actuelle est 'i', et que la meilleure accuracy était à 'last_best_epoch',
        # l'arrêt anticipé avec 'patience' se déclencherait à l'époque 'i' si
        # i - last_best_epoch >= patience.

        if (i - last_best_epoch) >= patience:
            # L'arrêt anticipé aurait eu lieu à l'époque 'i'.
            # La meilleure époque pour cette patience est celle avec la meilleure val_accuracy
            # dans l'historique *jusqu'à* l'époque précédant l'arrêt (époque i).
            # Cette meilleure époque est `last_best_epoch + 1`.
            best_epoch_idx_for_patience = last_best_epoch
            best_val_accuracy_for_patience = val_accuracy[best_epoch_idx_for_patience]
            best_val_loss_for_patience = val_loss[best_epoch_idx_for_patience]
            epochs_run_for_patience = i + 1 # L'époque à laquelle l'arrêt se produit

            results_summary[patience] = {
                'simulated_epochs_run': epochs_run_for_patience, # L'époque à laquelle l'arrêt s'est produit
                'best_epoch_in_simulated_run': best_epoch_idx_for_patience + 1, # L'époque avec la meilleure performance dans l'exécution simulée
                'best_val_accuracy_in_simulated_run': best_val_accuracy_for_patience,
                'best_val_loss_in_simulated_run': best_val_loss_for_patience
            }
            print(f"  Arrêt anticipé simulé à l'époque {epochs_run_for_patience}")
            print(f"  Meilleure époque pour patience {patience}: {best_epoch_idx_for_patience + 1}")
            print(f"  Meilleure val_accuracy: {best_val_accuracy_for_patience:.4f}")
            print(f"  Meilleure val_loss: {best_val_loss_for_patience:.4f}")

            # Passer à la prochaine valeur de patience car nous avons trouvé le résultat pour celle-ci
            break

        # Si la boucle se termine sans déclencher l'arrêt anticipé pour une valeur de patience,
        # cela signifie que l'entraînement a duré toute la durée de l'exécution avec la patience maximale.
        # Dans ce cas, le meilleur résultat pour cette patience est le meilleur résultat global
        # trouvé pendant toute l'exécution avec la patience maximale.
        if i == len(val_accuracy) - 1 and patience not in results_summary:
             best_val_accuracy_overall = max(val_accuracy)
             best_epoch_overall_idx = val_accuracy.index(best_val_accuracy_overall)
             best_epoch_overall = best_epoch_overall_idx + 1
             best_val_loss_overall = val_loss[best_epoch_overall_idx]

             results_summary[patience] = {
                'simulated_epochs_run': len(val_accuracy),
                'best_epoch_in_simulated_run': best_epoch_overall,
                'best_val_accuracy_in_simulated_run': best_val_accuracy_overall,
                'best_val_loss_in_simulated_run': best_val_loss_overall
            }
             print(f"  L'arrêt anticipé ne s'est pas déclenché pour patience {patience}.")
             print(f"  Meilleure époque (exécution globale): {best_epoch_overall}")
             print(f"  Meilleure val_accuracy (exécution globale): {best_val_accuracy_overall:.4f}")
             print(f"  Meilleure val_loss (exécution globale): {best_val_loss_overall:.4f}")


# Afficher le tableau récapitulatif
print("\n--- Tableau Récapitulatif des Résultats (Analyse de l'historique) ---")
# Convertir results_summary en DataFrame
results_df = pd.DataFrame.from_dict(results_summary, orient='index')
results_df.index.name = 'Patience'
# Renommer les colonnes pour plus de clarté
results_df = results_df.rename(columns={
    'simulated_epochs_run': 'Époques Exécutées (Simulé)',
    'best_epoch_in_simulated_run': 'Meilleure Époque (Simulé)',
    'best_val_accuracy_in_simulated_run': 'Meilleure Val Accuracy',
    'best_val_loss_in_simulated_run': 'Meilleure Val Loss'
})
display(results_df)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd # Make sure pandas is imported

# Assuming results_df is already created and populated in a previous cell

if 'results_df' in locals() and not results_df.empty:
    print("\n--- Visualisation des Résultats ---")

    # Plotting Best Validation Accuracy
    plt.figure(figsize=(10, 5))
    plt.plot(results_df.index, results_df['Meilleure Val Accuracy'], marker='o', linestyle='-', color='b')
    plt.title('Meilleure Validation Accuracy vs Patience')
    plt.xlabel('Patience')
    plt.ylabel('Accuracy')
    plt.xticks(results_df.index) # Set x-ticks to patience values
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Plotting Best Validation Loss
    plt.figure(figsize=(10, 5))
    plt.plot(results_df.index, results_df['Meilleure Val Loss'], marker='o', linestyle='-', color='r')
    plt.title('Meilleure Validation Loss vs Patience')
    plt.xlabel('Patience')
    plt.ylabel('Loss')
    plt.xticks(results_df.index) # Set x-ticks to patience values
    plt.grid(True)
    plt.tight_layout()
    plt.show()

else:
    print("Le DataFrame results_df n'existe pas ou est vide. Veuillez exécuter la cellule de calcul des résultats.")


In [ ]:
import pandas as pd

print("\n--- Tableau Récapitulatif des Résultats ---")
results_df = pd.DataFrame.from_dict(results_summary, orient='index')
results_df.index.name = 'Patience'
display(results_df)


In [ ]:
import os

for file in os.listdir('.'):
    if file.endswith('.h5'):
        size = os.path.getsize(file) / (1024 * 1024)
        print(f"{file:40s}  {size:8.2f} Mo")


In [ ]:
import os

# Répertoire où chercher les fichiers (. = répertoire courant)
folder = '.'

# Liste tous les fichiers .h5 du dossier (et sous-dossiers si tu veux)
h5_files = []
for root, dirs, files in os.walk(folder):
    for file in files:
        if file.endswith('.h5'):
            filepath = os.path.join(root, file)
            size_mb = os.path.getsize(filepath) / (1024 * 1024)  # taille en Mo
            h5_files.append((filepath, size_mb))

# Affichage formaté
if h5_files:
    print(f"{'Fichier':60s} | Taille (Mo)")
    print("-" * 80)
    for name, size in h5_files:
        print(f"{name:60s} | {size:10.2f}")
else:
    print("Aucun fichier .h5 trouvé dans le dossier courant.")


In [ ]:
t.stop()
